 
> **更新说明**：本版本基于 dbt Core 1.11 / Fusion Engine / 2025-2026 官方文档全面更新

- [Level 1 — 基础层（Junior 应掌握，Senior 默认已知）](#level-1)
- [Level 2 — Senior 核心竞争力](#level-2)
- [Level 3 — 了解即可（Staff/Principal 层）](#level-3)
- [面试高频问题 & 标准答法](#interview-qa)


[using dbt with snowflake](https://www.snowflake.com/en/developers/guides/data-teams-with-dbt-cloud/#dbt-configuration)
---


## Level 1 — Foundamental

<img src='./pic/1_dbt_in_data_pipeline.webp' width=700>

### 1.1 dbt 是什么

- dbt = **data build tool**
- 定位：专门处理数据仓库里的 **transformation** 层（T in ELT）
- 它不负责把数据从源头搬进来（那是 Fivetran / Airbyte / Kafka 的事）
- 它<u>只在data warehouse/lakehouse 内部运行 SQL，把 raw data 变成可用的 analytical tables</u>
- dbt 是一个 **SQL transformation framework**
	- 把你写的 model（SQL + Jinja）**编译成纯 SQL**
	- 然后通过 adapter 发送到数据平台执行
- 底层原理：把你写的 `.sql` 文件编译成目标平台（Snowflake / BigQuery / Databricks）的 SQL，然后由目标平台执行计算
  - 真正跑计算的是这些平台的 compute engine，不是 dbt
- 当前有两个引擎：
  - **dbt Core**（Python，开源；只支持 SQL 的传统 dbt）
  - **dbt Fusion**（Rust，2025 年发布，更快；支持 SQL + Python 的新一代 dbt，统一开发体验）

**实际执行流程:**   

```text
Raw Data (已经在仓库里)
      ↓
  1. 你写 dbt models (SQL + Jinja)
      ↓
  2. dbt compile → 生成纯 SQL
      ↓
  3. dbt run：
	•	把 SQL 发给目标数据仓库
	•	由数据仓库执行计算
      ↓
Clean / Curated Tables
```

#### 为什么会有 dbt Fusion？

| Dimension | dbt Core | dbt Fusion |
|----------|---------|------------|
| Language | SQL (+ Jinja) only | SQL + Python |
| Execution Model | SQL pushdown to warehouse | Hybrid (SQL + Python execution) |
| Python Support | ❌ None | ✅ Native support |
| Use Cases | Pure SQL transformations | ML / advanced transformations |
| Engine | SQL compiler only | Multi-language engine |
| Flexibility | Medium | High |
| Learning Curve | Lower | Higher |
| Maturity | Very mature | New / evolving |

👉 因为 dbt Core 有一个很大的限制：
- 很多复杂逻辑：
  - 很难用 SQL 写（比如 ML / graph / complex logic）

所以 dbt Fusion 的目标是：

“让数据工程师在一个框架里同时用 SQL 和 Python”


#### dbt core vs dbt cloud

| Dimension | dbt Core | dbt Cloud |
|----------|---------|----------|
| Type | Open-source **CLI tool** | Managed SaaS **platform** |
| Execution | Runs locally or on your own infra | Runs on managed cloud |
| Scheduler | ❌ Not included (use Airflow / cron) | ✅ Built-in scheduler |
| UI | ❌ None | ✅ Web UI (lineage, logs, runs) |
| CI/CD | ❌ Self-managed (e.g., GitHub Actions) | ✅ Built-in CI |
| Monitoring & Logs | ❌ Self-managed | ✅ Integrated observability |
| Access Control | ❌ Minimal | ✅ RBAC & enterprise features |
| Setup Effort | High (manual setup) | Low (out-of-the-box) |
| Flexibility | High | Medium |
| Cost | Free | Paid |
| Best For | Teams with existing infra | Teams wanting fast setup |

### 1.2 核心概念：model

- **model** = 一个 `.sql` 文件，每个文件对应一张表或视图
- **文件名就是表名**：`stg_accounts.sql` → 生成 `stg_accounts` 这张表
- dbt 会自动处理 `CREATE TABLE AS SELECT` 的样板代码，你只需要写 `SELECT`

```sql
-- models/staging/stg_accounts.sql
SELECT
    account_id,
    account_name,
    created_at
FROM {{ source('salesforce', 'accounts') }}
```



### 1.3 `ref()` 和 `source()`


- **`ref('model_name')`**：引用**另一个 dbt model（内部依赖）**
  - 作用：dbt 根据 `ref()` 自动推断 DAG（有向无环图），决定执行顺序
  - 好处：不需要硬编码 schema 名，跨环境自动解析（dev/prod 用不同 schema）
- **`source('source_name', 'table_name')`**：引用 raw data（数据仓库里已有的表）
  - 需要在 `sources.yml` 里声明
  - 好处：集中管理 raw table 位置，改一处全局生效

`ref()` 管的是 dbt 内部 model 之间的依赖，`source()` 管的是 dbt 和外部 raw data 之间的连接点。

```sql
-- ❌ 硬编码（不可移植）
FROM raw.salesforce.accounts

-- ✅ 用 source()
FROM {{ source('salesforce', 'accounts') }}

-- ✅ 用 ref()
FROM {{ ref('stg_accounts') }}
```



#### `ref()` 解释
`ref()` 解决的是"dbt 怎么知道跑的顺序"和"同一份代码怎么在 dev/prod 都能跑"这两个问题。  

假设有两个 model：`stg_accounts` 和 `dim_accounts`，其中 `dim_accounts` 依赖 `stg_accounts`。  

**如果不用 ref()，硬编码写法**：    
```sql
-- models/marts/dim_accounts.sql
SELECT * FROM analytics_prod.staging.stg_accounts
```
这有两个问题：
1. **执行顺序问题**。你有 200 个 model，dbt 怎么知道先跑哪个、后跑哪个？
   - 如果它先跑了 `dim_accounts`，但 `stg_accounts` 还没跑完，查到的就是<u>旧数据</u>。
   - **硬编码 SQL 里 dbt 看不出谁依赖谁，所以它无法自动排序**。
   - 用了 `ref('stg_accounts')` 之后，dbt 扫描所有 model 文件，发现 `dim_accounts` 里引用了 `stg_accounts`，就自动画出一条依赖边：`stg_accounts` → `dim_accounts`。所有这些边组合起来就是 DAG，**dbt 按 DAG 的拓扑顺序执行，保证上游一定先于下游完成**。
2. **跨环境问题**。
   - 你<u>本地开发</u>时，表在 `analytics_dev.yours.stg_accounts`；
   - <u>CI 环境</u>可能在 `analytics_ci.pr_123.stg_accounts`；
   - <u>生产环境</u>是 `analytics_prod.staging.stg_accounts`。
   - 如果硬编码了 `analytics_prod.staging`，你在<u>dev</u> 里跑就会直接查<u>生产数据</u>，要么<u>报权限错误，要么污染生产表</u>。
   - 用 `ref('stg_accounts')` 之后，**dbt 在编译时根据当前 profiles.yml 里的 target 配置自动替换 schema**：
    ```sql
    -- dev 环境编译结果
    SELECT * FROM analytics_dev.yours.stg_accounts

    -- prod 环境编译结果
    SELECT * FROM analytics_prod.staging.stg_accounts
    ```
   - 你的 SQL 文件一行都不用改，**同一份代码在任何环境都能正确运行**。



#### `source()` 解释
`source()` 解决的是"raw table 的路径改了怎么办"和"怎么知道上游数据有没有按时到"这两个问题。

假设你的 Salesforce 数据通过 Fivetran 同步到仓库里，落在 `raw.salesforce.accounts` 这张表。你有 3 个 staging model 都需要读这张表。

**如果不用 source()，硬编码写法**：
```sql
-- models/staging/stg_accounts.sql
SELECT * FROM raw.salesforce.accounts

-- models/staging/stg_contacts.sql
SELECT * FROM raw.salesforce.contacts

-- models/staging/stg_opportunities.sql
SELECT * FROM raw.salesforce.opportunities
```

现在有一天，数据团队决定把 Salesforce 的 raw data 从 `raw.salesforce` 搬到 `raw_fivetran.salesforce_prod`（比如换了同步工具，或者做了 schema 重组）。你需要去每一个引用了这张表的 SQL 文件里手动改路径。3 个文件还好，如果是 30 个呢？漏改一个就是线上 bug。

**用 source() 的写法**： 
先在 YAML 里集中声明一次：
```yaml
# models/staging/_sources.yml
sources:
  - name: salesforce
    database: raw
    schema: salesforce
    tables:
      - name: accounts
      - name: contacts
      - name: opportunities
```
然后所有 model 里这样引用：
```sql
-- models/staging/stg_accounts.sql
SELECT * FROM {{ source('salesforce', 'accounts') }}
```
搬迁的时候，你只需要改 YAML 里的一处：
```yaml
sources:
  - name: salesforce
    database: raw_fivetran          # 改这里
    schema: salesforce_prod         # 改这里
```
30 个引用了这个 source 的 **model 一个都不用动，dbt 编译时自动解析到新路径**。  

另外 `source()` 还有一个 `ref()` 没有的能力：**freshness 检查**。你可以在 YAML 里配置：  
```yaml
sources:
  - name: salesforce
    tables:
      - name: accounts
        loaded_at_field: _fivetran_synced
        freshness:
          warn_after: {count: 12, period: hour}
          error_after: {count: 24, period: hour}
```
跑 dbt source freshness 就能**检测 raw data 是不是按时到了**。如果 Fivetran 同步挂了超过 24 小时，dbt 会报错，你就能在 transformation 之前就发现上游出了问题。


### 1.4 Materialization（物化策略）

- Materialization 解决的核心问题是：dbt 跑完你写的 SELECT 语句之后，结果放在哪、以什么形式存在？   
- 你写的每个 dbt model 本质上就是一条 SELECT 语句。但 SELECT 本身只是"查询"，它不会把结果存下来。
- Materialization 就是告诉 dbt：这个查询的结果，你要怎么"落地"到仓库里。

| Materialization | 底层行为 | 适用场景 |
|---|---|---|
| `view` | 每次查询时执行 SQL | 轻量 staging 层，不需要存储 |
| `table` | 每次 `dbt run` 都全量重建 | 数据量小、逻辑简单 |
| `incremental` | 只处理新数据，追加或 merge | 数据量大、历史数据稳定 |
| `ephemeral` | 编译成 CTE，不生成实体表 | 中间步骤，不需要单独查询 |
| `snapshot` | 记录源表随时间的变化（SCD Type 2） | 维表历史追踪 |

**设置方式**（在 model 文件顶部）：

```sql
{{ config(materialized='incremental', unique_key='event_id') }}

SELECT ...
```



#### View — 不存数据，存查询本身
```sql
CREATE VIEW stg_accounts AS
SELECT account_id, account_name, created_at
FROM raw.salesforce.accounts
```
dbt 实际上只是在仓库里创建了一个 view。View **不占存储空间**，它只是一个"保存的查询"。每次有人查 stg_accounts 的时候，仓库才临时执行里面的 SQL，实时计算出结果。 

- 好处是零存储成本，
- 坏处是**每次查都要重新算**。
- 适合 staging 层这种**逻辑简单、上游数据量不大**的场景。



#### Table — 把结果实实在在存下来
```sql
CREATE OR REPLACE TABLE dim_accounts AS
SELECT account_id, account_name, created_at
FROM stg_accounts
```
每次 `dbt run`，dbt 都会**先把旧表删了（DROP）再重建**，把 SELECT 的**结果完整写入一张新的物理表**。数据实实在在**存在磁盘上**，查询的时候**直接读，不需要重新计算**。  

- 好处是**查询快**（数据已经存好了），
- 坏处是**每次都全量重建**。如果你的表有 1 亿行，即使今天只新增了 1000 行，dbt 也会把 1 亿行全部重算一遍再存下来。
- 数据量小的时候无所谓，数据量大了就不可接受了。



#### Incremental — 只处理新数据
这就是为了解决 table 全量重建太贵的问题。
```sql
-- 首次运行：和 table 一样，全量建表
CREATE TABLE fct_events AS SELECT ... FROM raw_events

-- 后续运行：只插入/更新新数据
MERGE INTO fct_events AS target
USING (SELECT ... FROM raw_events WHERE event_ts > '上次跑到的最大时间') AS source
ON target.event_id = source.event_id
WHEN MATCHED THEN UPDATE ...
WHEN NOT MATCHED THEN INSERT ...
```
第一次跑的时候，和 table 一模一样，全量建表。之后每次跑，dbt 只处理"新来的"或"变化的"数据，然后追加或合并到已有的表里。一张 1 亿行的表，每天只新增 1000 行的话，incremental 只处理这 1000 行。  

- 好处是**快、省钱**。
- 坏处是**逻辑复杂**:  
  - 你要告诉 dbt **怎么判断"哪些是新的"**（通过 `is_incremental()` + watermark），
  - **怎么处理重复数据**（通过 `unique_key`），
  - 以及**用什么方式合并**（append / merge / delete+insert / microbatch）。



#### Ephemeral/ɪ'femərəl/ — 不生成任何表，只是一段可复用的 SQL 片段
```sql
-- dbt 不会在仓库里创建任何东西
-- 而是把这段 SQL 作为 CTE 嵌入到引用它的 model 里
WITH stg_currency AS (
    SELECT currency_code, exchange_rate FROM raw.currency_rates
)
```
假设你有一段**清洗逻辑**，3 个 model 都要用，但它本身不值得单独存成一张表。Ephemeral 就是把它变成一个 **CTE（Common Table Expression）**，编译时直接"内联"到调用者的 SQL 里。  

仓库里看不到这张表，dbt docs 的 DAG 里也不会单独显示。  

适合**纯粹的中间过渡逻辑，不需要被任何人直接查询**的场景。



#### Snapshot — 专门记录数据随时间的变化
其他四种 materialization 都是"当前状态"——每次跑完，表里存的是最新的数据。Snapshot 不同，它**会保留历史版本**。  

比如一个 account 的 industry 从 "Tech" 变成了 "Finance"，普通 table 里只能看到 "Finance"。Snapshot 表里会保留两行：一行是 "Tech"（有效期 1月-3月），一行是 "Finance"（3月至今）。这就是 **SCD Type 2**:  
```sql
{% snapshot user_snapshot %}
{{
  config(
    target_schema='snapshots',
    unique_key='user_id',
    strategy='timestamp',
    updated_at='updated_at'
  )
}}
SELECT * FROM source_table
{% endsnapshot %}
```



### 1.5 YAML 属性文件 — 文档 + 测试
In dbt, YAML files are used to **define metadata, tests, documentation, and configurations** for models and other resources.
- YAML 不参与计算！只负责定义规则 + 提供元数据 + 驱动 dbt 行为
  - ❌ 不会执行 SQL
  - ❌ 不做 transformation


- 每个目录下可以放任意名称的 `.yml` 文件（如 `_models.yml`、`_sources.yml`），dbt 会**自动扫描**
  - 注：`schema.yml` 只是历史惯例命名，dbt 对文件名没有限制
- 从 dbt Core v1.5 起，`version: 2` 是可选的，可以不写
- 从 dbt Core v1.8 起，YAML key 从 `tests:` 更名为 **`data_tests:`**（以区别于新增的 unit tests）
  - `tests:` 仍可用作别名，但同一 resource 上不能同时出现 `tests:` 和 `data_tests:`
- 内置 4 种 generic data tests：`not_null`、`unique`、`accepted_values`、`relationships`

```yaml
# models/staging/_stg_models.yml
models:
  - name: stg_accounts
    description: "Cleaned accounts from Salesforce"
    columns:
      - name: account_id
        description: "Primary key"
        data_tests:
          - not_null
          - unique
      - name: account_status
        data_tests:
          - accepted_values:
              values: ['active', 'inactive', 'churned']
```



#### 主要作用
1. **定义模型**（models）
    
    ```yaml
    models:
      - name: dim_users
        description: User dimension table
    ```
    👉 给表加说明（data catalog / 文档）

2. **定义 source**（数据来源）
    ```yaml
    sources:
      - name: raw
        tables:
          - name: users
    ```
👉 标记原始数据（方便 **lineage + test**）

3. **定义数据测试**（💥重点）
    ```yaml
    columns:
      - name: user_id
        data_tests:
          - not_null
          - unique
    ```
    
    👉 自动做数据质量检查, 常见测试：
      - not_null
      - unique
      - accepted_values
      - relationships

4. 定义关系（非常加分）
    ```yaml
    - name: user_id
      data_tests:
        - relationships:
            to: ref('dim_users')
            field: user_id
    ```
  👉 类似 foreign key constraint（但在分析层）

5. 配置（config）
    ```yaml
    config:
      materialized: table
    ```
    👉 控制 table / view / incremental

#### data_tests vs unit_tests
Data tests check whether the data violates constraints,
while unit tests check whether the transformation produces the expected result.

| Dimension | data_tests | unit_tests |
|----------|-----------|------------|
| 核心目的 | 数据质量（data correctness） | 逻辑正确性（logic correctness） |
| 数据来源 | 真实表（production data） | mock 数据（given） |
| 测试方式 | constraint check | result comparison |
| SQL 本质 | WHERE + COUNT | EXCEPT / set comparison |
| 是否依赖数据规模 | ✅ 是（大数据） | ❌ 否（小数据） |
| 使用时机 | 数据产出后 | 开发阶段 / CI |

In dbt, unit tests are defined declaratively by specifying input data and expected output for a model.  

so one unit_tests includes:  
- model（测试哪个模型）
- given（输入数据）
- expect（期望输出）

dbt unit test 的底层是怎么实现的（SQL 角度）: unit tests are implemented by materializing mock input data (本质是转换成 CTE 或临时表), running the model SQL, and comparing the result with expected output using SQL set operations.

```yaml
unit_tests:
  - name: test_user_transformation
    model: ref('dim_users')

    given:
      - input: ref('raw_users')
        rows:
          - {user_id: 1, age: 17}
          - {user_id: 2, age: 25}

    expect:
      rows:
        - {user_id: 2, is_adult: true}
```

👉 unit test 适用于：
- 复杂 SQL 逻辑
- CASE WHEN / business rules
- joins / filtering logic
- 边界条件（edge cases）

🚫 不适合的场景:  
- ❌ 检查 null / unique → 用 data_tests   
- ❌ 大规模数据验证  → use Data Quality Framework: Great Expectations, Soda    
- ❌ 数据质量监控 → Monte Carlo, Datadog


### 1.6 seed
- In dbt, seeds are static CSV files that are loaded into the data warehouse as tables.
- Seeds are used for **small, static datasets** that **don’t require transformation** but need to live in the warehouse.
- 比如有一个 countries.csv 表，run `dbt seed`，就会在仓库里生成 countries 一张表
- 常见用途：
  - 小型维表（lookup table），e.g.  |country_code | country_name|, 不需要 ETL，直接 seed
  - 映射表（mapping），e.g. 状态码 → 描述, category → group
  - 单元测试数据（💥重点），给 unit_tests 提供输入数据
  - 固定 reference data， 配置数据，常量表

seed vs model  

| 维度| seed| model| 
| ---| ----| ----| 
| 来源| CSV 文件| SQL| 
| 是否计算| ❌ 不计算| ✅ transformation| 
| 数据类型| 静态数据| 动态数据| 
| 使用场景| lookup / config| business logic| 


### 1.7 常用命令

```bash
dbt run                        # 运行所有 models
dbt run --select stg_accounts  # 只运行指定 model
dbt run --select +stg_accounts # 运行 stg_accounts 及其所有上游依赖
dbt run --select stg_accounts+ # 运行 stg_accounts 及其所有下游依赖

dbt build                      # ⭐ 按 DAG 顺序交错运行 seed → snapshot → run → test
                               #    test 失败会阻断下游 model，防止坏数据传播
dbt build --select marts.core  # 只 build 指定 scope

dbt test                       # 运行所有测试（data tests + unit tests）
dbt test --select stg_accounts # 只测试指定 model
dbt test --select "test_type:unit"  # 只跑 unit tests（v1.8+）
dbt test --select "test_type:data"  # 只跑 data tests（v1.8+）

dbt retry                      # ⭐ 只重试上次运行中失败的 nodes

dbt docs generate              # 生成文档
dbt docs serve                 # 本地启动文档网站（含 DAG 可视化）

dbt debug                      # 检查连接配置
dbt compile                    # 只编译 SQL，不执行（用于 debug Jinja）
```

**`dbt build` vs `dbt run` + `dbt test` 的区别**：
- `dbt build` 在 DAG 中交错执行 run 和 test，而不是先全部 run 再全部 test。
- 如果一个 model 的测试失败了，它的下游 model 不会被运行，提前阻断坏数据向下传播。
- 面试中更推荐说 `dbt build` 而非分开的 `dbt run` + `dbt test`。

---


## Level 2 - Senior 
核心竞争力

### 2.1 Incremental Models（面试最高频）
<img src='./pic/1_dbt_regular_incremental_models.png' width=500>

#### 核心机制

```sql
{{ config(
    materialized='incremental',
    unique_key='event_id',
    incremental_strategy='merge'
) }}

SELECT
    event_id,
    account_id,
    event_type,
    event_ts,
    processed_at
FROM {{ source('kafka_events', 'raw_events') }}

{% if is_incremental() %}
  -- 只有在 incremental 模式下才加这个 WHERE 条件
  WHERE event_ts >= (SELECT MAX(event_ts) FROM {{ this }})
{% endif %}
```

- **`is_incremental()`**：
  - 是 dbt 提供的 Jinja 函数
  - 首次运行返回 `False`（全量），后续运行返回 `True`（增量）
  - True → 当前 model 是 incremental 模式运行
  - False → full-refresh 或第一次 build
  - 作用：只有在 incremental run 时才执行里面的 SQL
- **`{{ this }}`**：
  - 引用当前 model 本身（已存在的那张表），用于查 watermark
  - 意思：只处理大于已有最大时间戳的新数据，避免重复插入历史数据

🚫 常见误区
- ❌ “WHERE 条件决定是否 incremental”
  - 实际是 incremental 模式决定是否执行 WHERE
- ❌ incremental 模式就自动只插入新数据
  - 必须写条件（如时间戳）保证增量

#### Incremental Strategy 对比

| Strategy | 行为 | 适用场景 |
|---|---|---|
| `append` | 直接 INSERT，**不去重** | 事件流，event_id 天然唯一 |
| `merge` | MERGE INTO，**按 unique_key 更新或插入** | 维表更新，数据可能变化 |
| `delete+insert` | 先按 partition 删除，再插入 | 按日期分区的大表 |
| `insert_overwrite` | 覆盖整个 partition（Spark/Databricks 专用） | Databricks Delta Lake |
| **`microbatch`** | **按时间切 batch，每 batch 独立 SQL（v1.9+）** | **大规模时间序列，详见 2.2** |

<img src='./pic/1_incremental_stratey_decision_tree.webp' width=700>

```text
数据会被更新吗？
├── 不会（immutable events）
│   ├── 数据量小 → append
│   └── 数据量极大 + 有时间列 → microbatch
└── 会（CDC / 维表变更）
    ├── 需要行级 upsert → merge
    └── 按日期分区，整 partition 替换
        ├── Databricks → insert_overwrite
        └── 其他平台 → delete+insert
```

1. `Append` — 直接 INSERT，不去重   
   - just inserts the selected records into the destination table. 
   - It can’t update or delete records, just insert. 
   - is suitable when **duplicates are not a concern**.
   - 适用场景：事件流，event_id 天然唯一，数据不可变（immutable events）
   - Pros：最快，最简单，无行级比较开销，写入性能最高
   - Cons：完全不处理重复——如果 Kafka replay 或 CDC at-least-once 导致重复记录，会直接写入两份；也不能更新已有记录
   
   <img src='./pic/1_incremental_append.webp' width=650>

2. `Merge` — MERGE INTO，按 unique_key 更新或插入
   - solves the problem of duplicate records. 
   - If the **unique key already exists** in the destination table, the merge will **update the record**. 
   - And if the records **don’t exist**, merge will **insert** them.
   - 适用场景：dimention table更新（如 account 的 industry、tier 变化），数据可能被修改
   - Pros：同时处理 INSERT 和 UPDATE，保证 unique_key 不重复，一条 SQL 搞定 upsert
   - Cons：行级比较开销大, 对于超大表（100M+ 行），MERGE 需要逐行匹配 unique_key，可能非常慢；在某些平台上（如 BigQuery）MERGE 有并发写限制
   
   <img src='./pic/1_incremental_merge.webp' width=650>

   **Merge with clustered**     
   - To check if the unique key of both tables matches, merge has **to scan the whole destination table**. Performing this full scan can be very costly. 
   - The **destination table can be clustered** to increase the merge performance and reduce costs. 
   - It will scan the clustered destination table, instead of full table

3. `Delete+insert` — 先按 partition 删除，再插入
   - is very similar to merge, but instead of updating existing records and inserting new records, 
   - it **deletes existing records and inserts both** new and existing records.
   - 适用场景：按日期分区的大表，历史分区数据可能被修正
   - Pros：比 merge 快, 直接按 partition 批量删除再插入，避免逐行匹配；在不支持 MERGE 或 MERGE 性能差的平台上是好替代
   - Cons：删除和插入之间存在短暂窗口，期间查询可能看到不完整数据（非原子操作，部分平台除外）；需要明确指定 partition key
   - 何时选 delete+insert 而非 merge：当你按日期分区且每个分区内数据量大时，delete+insert 比 merge 快得多，因为它跳过了逐行匹配
   
   <img src='./pic/1_incremental_delete_insert.webp' width=650>

4. `Insert_overwrite` (with partitioned)
   - **solves the problem of a full scan**. 
   - The solution used by insert overwrite is to **work with partitions**. 
   - the partition can be of the following types: 
     - date, datetime, timestamp, int64. 
     - But if the timestamp is skewed, Insert+overwrite is not an ideal solution.
   - The insert overwrite strategy **deletes the selected partitions** from the current destination table and inserts the selected transformed partitions into it.
   - it **can generate duplicates** if you do not set it right. 
     - A periodic full refresh solves this problem, 
     - but if you can’t wait for the entire refresh to run, you should use another column or consider using the merge strategy
   - 适用场景：Databricks Delta Lake 上按日期分区的聚合表
   - Pros：原子操作, 整个 partition 被一次性替换，没有 delete+insert 的不完整数据窗口问题；利用 Delta Lake 的 partition overwrite 语义，性能好
   - Cons：只在 Spark/Databricks 上可用，不跨平台；只能按 partition 粒度替换，不能做行级更新
  
   <img src='./pic/1_incremental_insert_overwrite.webp' width=650>

5. `microbatch` — 按时间切 batch，每 batch 独立 SQL（v1.9+）
   - 适用场景：大规模时间序列（100M+/天的事件数据），需要可靠 backfill，详见 2.2 节
   - Pros：每个 batch 独立幂等，失败只重跑该 batch 而非整个 model；原生支持 lookback 和日期范围 backfill（`--event-time-start`/`--event-time-end`）；不需要手写 `is_incremental()` 逻辑；支持并行执行
   - Cons：需要可靠的时间列（`event_time`）；上游 model 也必须配置 `event_time`，否则每个 batch 全表扫描上游；v1.9+ 才可用，并行执行对 adapter 有要求（Snowflake 最成熟）

[dbt Incremental: Implementing & Testing](https://medium.com/refined-and-refactored/dbt-incremental-implementing-testing-p2-967e8a8e4240)



#### Late-arriving Data 问题（高频追问）

**问题**：上游数据延迟，事件 timestamp 是 3 天前的，但今天才到仓库

**解决方案：lookback / reprocessing window**  

In dbt, late-arriving data is handled by incremental models combined with conditional filters or reprocessing windows, sometimes using `delete+insert` or `insert_overwrite` strategies.

```sql
{% if is_incremental() %}
  WHERE event_ts >= DATEADD(day, -3, (SELECT MAX(event_ts) FROM {{ this }}))
{% endif %}
```

- 每次多往回看 3 天，重新处理这个窗口内的数据
- 代价：多处理一些历史数据
- 收益：不会漏掉 late-arriving records
- e.g. Kafka 消费延迟 + 网络问题可能导致事件晚到，这个 pattern 是标配



#### `on_schema_change` 配置
It is a configuration for incremental models that tells dbt **how to react when the table schema changes** (e.g., new/missing columns).  

It enables incremental models to **evolve with source table schema changes without failing**, but it must be used carefully to avoid silent data loss.

| 值 | 行为 |
|---|---|
| `ignore`（默认） | 忽略 schema 变化，可能丢列 |
| `fail` | schema 变化时直接报错 |
| `append_new_columns` | 自动加新列，不删旧列 |
| `sync_all_columns` | 完全同步（加新列、删旧列、改类型） |

> ⚠️ Snowflake 计划在 2026 年 5 月变更 string/binary 类型默认列大小。使用 `sync_all_columns` + dbt-snowflake < v1.10.6 可能导致 incremental model 构建失败。

```yaml
models:
  - name: dim_users
    materialized: incremental
    config:
      on_schema_change: append_new_columns
```

Keys:  
1. **只影响 incremental 模型**
   - full-refresh / table materialization 不受影响
2. 处理新增列 vs 删除列区别
   - append_new_columns 只添加，不删除
   - 删除列还是会报错
3. 安全性考虑
   - 对生产大表，通常先全量刷新再改 schema，避免增量覆盖风险


#### Incremental vs Table — 什么时候切换？

| 场景 | 建议 |
|---|---|
| 表很小（< 1M 行）| 用 `table`，逻辑简单可靠 |
| 历史数据不变，只有新增 | 用 `incremental` + `append` |
| 历史数据会被更新（如 CDC）| 用 `incremental` + `merge` |
| 数据量极大（100M+/天）| 必须用 `incremental`，全量不现实 |
| 超大时间序列 + 需要可靠 backfill | 用 `microbatch`（v1.9+） |


### 2.2 ⭐ Microbatch Strategy（dbt Core v1.9+，面试新热点）

Microbatch 是 2024-2025 年 dbt 最重要的新 incremental strategy，**专为大规模时间序列数据设计**。 

Microbatch splits processing into **independent time-based batches** — each batch gets its own SQL query, can be retried independently, and dbt automatically handles the time-range filtering. The big wins are: no more manual `is_incremental()` logic, native lookback for late-arriving data, and the ability to backfill specific date ranges with `--event-time-start` and `--event-time-end` instead of a full refresh. It's available since dbt Core 1.9.

#### 核心思路

- 不再需要手写 `is_incremental()` 和 `{{ this }}` 来做 watermark
- dbt 自动按 `event_time` 列将数据切分为多个 batch（如按天/按小时），每个 batch 独立执行一条 SQL
- 每个 batch 是 idempotent，失败可以单独重试，不需要重跑整个 model
- 支持 parallel batch execution（多线程并行处理不同 batch）

```sql
{{ config(
    materialized='incremental',
    incremental_strategy='microbatch',
    event_time='event_ts',
    batch_size='day',
    lookback=3,        -- 自动往回多看 3 个 batch，处理 late-arriving data
    begin='2020-01-01',
    full_refresh=false
) }}

SELECT
    event_id,
    event_ts,
    account_id,
    event_type
FROM {{ ref('stg_email_events') }}
-- 不需要写 is_incremental() block！dbt 自动生成 WHERE 条件
```

#### 与传统 incremental 的关键区别

| 维度 | 传统 incremental | microbatch |
|---|---|---|
| 查询数量 | 1 条大 SQL | 多条小 SQL（每个 batch 一条） |
| `is_incremental()` | 手动写 | 不需要 |
| `{{ this }}` watermark | 手动查 | dbt 自动管理 |
| 失败重试 | 重跑整个 model | 只重跑失败的 batch |
| Backfill | 需要 `--full-refresh` 或 vars hack | `--event-time-start` / `--event-time-end` 原生支持 |
| Late-arriving data | 手动写 lookback window | `lookback` config 自动处理 |
| 并行执行 | 不支持 | 支持（`concurrent_batches` config） |

#### 重要限制

- 需要数据有可靠的时间列（`event_time`）
- **上游 model 也需要配置 `event_time`**，否则每个 batch 会全表扫描上游
- 目前不是所有 adapter 都支持并行执行（Snowflake 最成熟）
- `dbt retry` 对 microbatch 会使用原始调用时间重算 batch

#### Backfill 用法

```bash
# 只重跑 2025 年 1 月的数据，不影响其他日期
dbt run --select fct_events --event-time-start "2025-01-01" --event-time-end "2025-02-01"
```





### 2.3 Macros & Jinja
展示 Senior 深度的关键

#### 什么是 macro

- macro = dbt 里的函数，用 Jinja2 语法写
- 存放在 `macros/` 目录下
- 作用：封装重复逻辑，让整个团队调用同一套标准实现

#### 基础语法

```sql
-- macros/deduplicate.sql
{% macro deduplicate(relation, partition_by, order_by) %}
  SELECT *
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY {{ partition_by }}
        ORDER BY {{ order_by }} DESC
      ) AS _row_num
    FROM {{ relation }}
  )
  WHERE _row_num = 1
{% endmacro %}
```

**调用方式**：

```sql
-- models/staging/stg_events.sql
{{ deduplicate(
    relation=source('kafka', 'raw_events'),
    partition_by='event_id',
    order_by='processed_at'
) }}
```

#### SCD Type 2 Macro

**SCD Type 2 是什么**：
- Slowly Changing Dimension Type 2
- 当维表某个字段更新时（如 account 的 industry 变了），不覆盖旧记录
- 而是：把旧记录标记为 `is_current = False`，加上 `effective_end_date`；插入新记录标记 `is_current = True`

> **注意**：dbt 有内置的 **snapshot** 功能也能实现 SCD Type 2（见 2.7 节），且在 v1.9 做了重大升级。面试中应该知道两者的关系——自定义 macro 适合需要 **hash-based change detection** 和自定义 **surrogate key** 的高级场景，而**内置 snapshot 适合标准维表追踪**。

**完整 macro 实现**：

```sql
-- macros/scd_type_2.sql
{% macro scd_type_2(
    source_relation,
    unique_key,
    updated_at_col,
    tracked_columns
) %}

WITH source_data AS (
    SELECT
        {{ unique_key }},
        {% for col in tracked_columns %}
        {{ col }},
        {% endfor %}
        {{ updated_at_col }},
        -- 注意：实际 hash 函数因平台而异，面试中说 "hash-based" 比说 "MD5" 更准确
        MD5(CONCAT(
            {% for col in tracked_columns %}
            COALESCE(CAST({{ col }} AS STRING), '') 
            {% if not loop.last %}, '|', {% endif %}
            {% endfor %}
        )) AS _row_hash
    FROM {{ source_relation }}
),

existing_records AS (
    SELECT * FROM {{ this }}
    WHERE is_current = TRUE
),

-- 找出发生变化的记录（hash 不一致）
changed AS (
    SELECT s.*
    FROM source_data s
    LEFT JOIN existing_records e ON s.{{ unique_key }} = e.{{ unique_key }}
    WHERE e.{{ unique_key }} IS NULL           -- 新增记录
       OR s._row_hash != e._row_hash           -- 有字段变化
)

SELECT
    {{ dbt_utils.generate_surrogate_key([unique_key, updated_at_col]) }} AS surrogate_key,
    {{ unique_key }},
    {% for col in tracked_columns %}
    {{ col }},
    {% endfor %}
    {{ updated_at_col }} AS effective_start_date,
    NULL AS effective_end_date,
    TRUE AS is_current,
    _row_hash
FROM changed

{% endmacro %}
```

dimension tables like accounts and contacts change over time — account industry, tier, or owner gets updated. Rather than overwriting history, I built a dbt macro for SCD Type 2 that handles surrogate key generation via hash-based change detection, detects row changes by comparing a hash of tracked columns, closes out stale records by setting `is_current = False` and `effective_end_date`, and inserts new versions. The macro is parameterized so any team member can apply the same pattern with two lines of code. dbt also has built-in snapshots for standard SCD Type 2 — our macro adds custom hash-based detection and surrogate key generation on top of that.



### 2.4 dbt Project Structure

```text
my_dbt_project/
├── dbt_project.yml        # 项目配置文件（必须有，项目的"身份证"）
├── profiles.yml           # 连接配置（本地开发用，定义连到哪个仓库）
├── packages.yml           # 第三方包声明（如 dbt_utils, dbt_expectations）
│
├──models/                 # ⭐ 核心 — 所有 transformation 逻辑
├── staging/                # stg_xxx  — 1:1 对应 source table，只做 rename/cast/clean
├── intermediate/           # int_xxx  — 业务逻辑中间步骤，不直接暴露给消费者
├── marts/                  # 直接面向消费者（BI、ML、API）
│   ├── core/               # dim_xxx, fct_xxx — 核心维表和事实表
│   └── ml/                 # feature_xxx — ML feature tables
└── _sources.yml            # raw source 声明（文件名可以任意取）
│
├── tests/                 # 自定义 singular tests（单独的 .sql 测试文件）
├── macros/                # 可复用的 Jinja 函数
├── seeds/                 # 小型静态 CSV 文件（如 country code mapping）
├── snapshots/             # SCD Type 2 snapshot 定义
│
├── analyses/              # 纯分析用 SQL，dbt compile 会编译但不会执行
└── target/                # dbt 编译输出（自动生成，通常 gitignore）
```



#### `dbt_project.yml` 
是整个项目的入口。dbt 靠它识别"这是一个 dbt 项目"。里面定义项目名、默认 materialization、目录路径等全局配置：

```yaml
name: 'our_analytics'
config-version: 2

models:
  our_analytics:
    staging:
      +materialized: view       # staging 层默认用 view
    marts:
      +materialized: table      # marts 层默认用 table
```
#### `profiles.yml` 
定义 dbt 连哪个仓库。
- 本地开发时通常放在 ~/.dbt/profiles.yml（不在项目目录里，因为包含 credentials 不应该提交到 Git）。
- 在 Databricks Workflows 或 dbt Cloud 里，连接信息通过环境变量注入，不需要这个文件。




#### `models/` 每层职责

**Staging 层**：
- 只做机械性清洗：rename columns（snake_case）、cast types、filter deleted rows
- 绝对不做 JOIN，不做业务逻辑
- `materialized='view'`（轻量，不占存储）
- 命名：`stg_{source}_{table}` → 如 `stg_salesforce_accounts`

**Intermediate 层**：
- 处理跨 source 的 JOIN，或者复杂业务逻辑的中间步骤
- 不对外暴露（消费者不应直接查这层）
- `materialized='ephemeral'` 或 `view`

**Marts 层**：
- 直接面向消费者（BI、ML、API）
- `dim_xxx`：维表（accounts, contacts, users）
- `fct_xxx`：事实表（events, activities, opportunities）
- `materialized='table'` 或 `incremental`（数据量大时）

**为什么这样分层**:   

- **关注点分离**：staging 改动不影响 marts，marts 逻辑变了不需要动 raw source
- **可测试性**：每层都可以独立测试
- **可复用性**：intermediate 层逻辑可以被多个 mart 共享



#### other folders
`seeds/` 存放小型静态 CSV，
- dbt seed 会把它们加载到仓库里变成表。
- 适合很少变化的 lookup 数据，比如 country code mapping、部门名称对照表。
- 不适合大数据量, 超过几千行就应该走正常的 ingestion pipeline。  

`analyses/` 比较小众。
- 放一些"我想用 dbt 的 Jinja + ref() 编译能力，但不需要 dbt 把结果写到仓库里"的 SQL。
- 比如一次性的探索性分析，或者给业务方写的 ad-hoc query 模板。

`tests/` 放 singular tests
- 完整的 .sql 文件，返回"不应该存在的行"。
- **一次性的、专门针对某个业务场景的检查**。写一条 SELECT，返回的每一行代表一个"违规记录"。如果返回 0 行，测试通过；返回任何行，测试失败。
- 和 **YAML 里声明的 generic data tests（下面2.5所提到的tests）是互补的**。
- 比如你想检查"同一个 account 不应该同时出现在 active 和 churned 状态"，这种**复杂的业务逻辑校验**写成 singular test 更清晰:  
    ```sql
    -- tests/assert_no_account_in_both_active_and_churned.sql
    SELECT account_id
    FROM {{ ref('dim_accounts') }}
    WHERE account_status = 'active'
    AND account_id IN (
        SELECT account_id FROM {{ ref('dim_accounts') }}
        WHERE account_status = 'churned'
    )
    ```
- 它不接受参数，不能被其他 model 复用。如果另一张表也有类似需求，你得再写一个 SQL 文件。

面试中被问到 project structure，重点讲 models/ 的分层（staging → intermediate → marts）就够了。其他目录知道存在、能说清楚用途即可。

### 2.5 Data Quality Testing

#### 两种测试类型（v1.8+ 重要区分）

| 类型 | 测试什么 | 运行时机 | 输入数据 | YAML key |
|---|---|---|---|---|
| **Data Tests** | 数据质量（唯一性、非空、引用完整性） | model 构建之后 | 真实仓库数据 | `data_tests:` |
| **Unit Tests** | SQL 逻辑是否正确 | model 构建之前 | 手动定义的 mock 数据 | `unit_tests:` |

#### Data Tests — 四种 built-in generic tests

```yaml
columns:
  - name: event_id
    data_tests:
      - not_null          # 不为空
      - unique            # 唯一
  - name: status
    data_tests:
      - accepted_values:
          values: ['sent', 'opened', 'clicked', 'bounced']
  - name: account_id
    data_tests:
      - relationships:    # 外键完整性
          to: ref('dim_accounts')
          field: account_id
```

#### Unit Tests（v1.8+，面试必知）

用静态 mock 数据验证 SQL 转换逻辑，不需要真实数据：

```yaml
unit_tests:
  - name: test_revenue_calculation
    model: fct_revenue
    given:
      - input: ref('stg_orders')
        rows:
          - {order_id: 1, quantity: 2, unit_price: 10.00, discount: 0.1}
          - {order_id: 2, quantity: 1, unit_price: 50.00, discount: 0}
    expect:
      rows:
        - {order_id: 1, revenue: 18.00}
        - {order_id: 2, revenue: 50.00}
```

**运行方式**：
```bash
dbt test --select "test_type:unit"    # 只跑 unit tests
dbt test --select "test_type:data"    # 只跑 data tests
dbt test                              # 两种都跑
```

Since dbt 1.8, we now have two types of tests. 
- Data tests validate data quality after models run — uniqueness, not-null, referential integrity. 
- Unit tests validate SQL logic before models run — I define static inputs and expected outputs in YAML, and dbt checks my transformation logic against those. This gives us test-driven development in dbt, catching logic bugs before they hit production.

#### 自定义 Generic Test

```sql
-- tests/generic/assert_row_count_above_threshold.sql
{% test assert_row_count_above_threshold(model, threshold) %}

SELECT COUNT(*) AS row_count
FROM {{ model }}
HAVING COUNT(*) < {{ threshold }}

{% endtest %}
```

**使用**：

```yaml
models:
  - name: fct_email_events
    data_tests:
      - assert_row_count_above_threshold:
          threshold: 1000000  # 每天至少 100 万行，否则报警
```



#### `store_failures` — 把失败写入表（高级用法）

```yaml
# dbt_project.yml
tests:
  +store_failures: true
  +schema: test_failures    # 失败记录写入这个 schema
```

- 好处：不只是知道"测试失败了"，还能**查看哪些具体行失败了**
- e.g. 发现某个 account_id 出现了 null，可以直接去 `test_failures` 表里看是哪些 event 受影响



#### dbt tests vs Great Expectations

| 维度 | dbt data tests | dbt unit tests | Great Expectations |
|---|---|---|---|
| 测试对象 | 数据质量（model 输出） | SQL 逻辑正确性 | 任意数据，包括 raw data |
| 运行时机 | `dbt test`，在 transformation 之后 | `dbt test`，在 transformation 之前 | 可在 ingestion 时就运行 |
| 配置方式 | YAML，轻量 | YAML + mock data | Python，灵活但复杂 |
| 适合场景 | 唯一性、引用完整性 | 复杂计算、edge cases | Pipeline-level 数据契约 |

dbt test + GE pattern:  
- **Unit tests** validate **SQL logic before models build** 
  - things like revenue calculations and conditional branches. 
- **Data tests** validate **output quality** 
  - uniqueness, not-null, referential integrity on our Gold layer. 
- **Great Expectations** sits earlier in the pipeline as data contracts at the **ingestion boundary**, catching schema drift and distribution anomalies **before bad data even reaches dbt**. 


### 2.6 dbt Contracts（v1.5+，多团队协作利器）

dbt 支持在 model 上定义 **contract**，强制指定列名、数据类型、约束：

```yaml
models:
  - name: dim_accounts
    config:
      contract:
        enforced: true          # here
    columns:
      - name: account_id
        data_type: bigint
        constraints:
          - type: not_null
          - type: primary_key
      - name: account_name
        data_type: varchar(256)
```

当 `contract.enforced: true` 时，如果 model 输出的 schema 与声明不符，dbt 会直接报错。这在多团队协作中非常有价值, **上游 model 的 schema** 成为一个 **Explicit commitment**。

e.g. We use dbt contracts on our Gold layer models to enforce a strict schema agreement. If anyone changes a mart model in a way that drops or renames a column, the build fails immediately instead of silently breaking downstream BI dashboards or ML feature pipelines.



### 2.7 Snapshots — dbt 内置 SCD Type 2（v1.9 重大升级）

dbt 有内置的 **snapshot** 功能，专门用于追踪source table随时间的变化，实现 SCD Type 2。

#### v1.9 之后推荐的 YAML 配置方式

```yaml
# snapshots/accounts_snapshot.yml
snapshots:
  - name: accounts_snapshot
    relation: source('salesforce', 'accounts')
    config:
      unique_key: account_id
      strategy: timestamp
      updated_at: updated_at
      dbt_valid_to_current: "timestamp '9999-12-31'"  # 当前记录用固定日期而非 NULL
      hard_deletes: invalidate                          # 删除记录的处理方式
```

> 注：从 v1.9 起，在 `.sql` 文件中用 `{% snapshot %}` Jinja block 定义 snapshot 是 **legacy 方式**。
> 新 snapshot 推荐用 YAML 配置。

#### Snapshot 策略

| 策略 | 检测方式 | 适用场景 |
|---|---|---|
| `timestamp` | 通过 `updated_at` 列判断变化 | 有可靠时间戳的表（推荐） |
| `check` | 比较指定列的值 | 没有可靠时间戳的表 |

#### `hard_deletes` 配置（v1.9+）

| 值 | 行为 |
|---|---|
| `ignore`（默认） | 忽略删除 |
| `invalidate` | 将已删除记录标记为无效（设置 `dbt_valid_to`） |
| `new_record` | 插入一条新记录并标记 `dbt_is_deleted = True` |

#### `dbt_valid_to_current` 配置（v1.9+）

默认情况下，当前有效记录的 `dbt_valid_to` 为 NULL。设置此配置后可以改为固定日期（如 `9999-12-31`），便于下游做 date range 过滤。

#### `snapshot_meta_column_names`（v1.9+）

可自定义 `dbt_valid_from`、`dbt_valid_to` 等元数据列的名称，对齐团队命名规范。

#### 自定义 Macro vs 内置 Snapshot — 什么时候用哪个

| 需求 | 推荐方案 |
|---|---|
| 标准维表历史追踪 | 内置 snapshot（简单、维护少） |
| 需要 hash-based change detection | 自定义 macro（更灵活） |
| 需要自定义 surrogate key 逻辑 | 自定义 macro |
| 需要跟踪特定列子集的变化 | 内置 snapshot `check` 策略 或自定义 macro |

- dbt has built-in snapshots for SCD Type 2, which got a major upgrade in v1.9 with YAML configuration, hard_deletes handling, and customizable meta columns. 
- We could also build a custom SCD Type 2 macro if our use case required hash-based change detection on specific tracked columns and custom surrogate key generation — capabilities that go beyond what the built-in snapshot provides out of the box. 
- But for simpler dimension tracking, the native snapshot is the recommended approach.


### 2.8 与 Orchestration 集成（Airflow / Databricks Workflows）

#### 在 Airflow 里调用 dbt

```python
# Airflow DAG
from airflow.operators.bash import BashOperator

dbt_build = BashOperator(
    task_id='dbt_build_marts',
    bash_command='dbt build --select marts.core --profiles-dir /etc/dbt',
)
```

#### 在 Databricks Workflows 里调用 dbt

- 迁移到 Databricks Workflows 后，**dbt 作为一个 Task 嵌入 Job**
- 通常用 `dbt-databricks` adapter，直接在 Databricks cluster 上运行
- Workflow Task 类型选 "dbt"，指定 `profiles.yml` 和要运行的 commands
- 2025 年新增：Databricks Lakeflow 中 dbt platform Task Type 进入 beta，提供更好的可观测性

**关键配置（`profiles.yml`）**：

```yaml
our_dbt:
  target: prod
  outputs:
    prod:
      type: databricks
      host: "{{ env_var('DATABRICKS_HOST') }}"
      http_path: "{{ env_var('DATABRICKS_HTTP_PATH') }}"
      token: "{{ env_var('DATABRICKS_TOKEN') }}"
      schema: gold
      catalog: outreach_prod    # Unity Catalog
```


### 2.9 dbt 与 Delta Lake / Databricks 的深度集成

#### `insert_overwrite` 策略（Databricks 专用）

```sql
{{ config(
    materialized='incremental',
    incremental_strategy='insert_overwrite',
    partition_by=['date_day']
) }}

SELECT
    DATE(event_ts) AS date_day,
    account_id,
    COUNT(*) AS event_count
FROM {{ ref('stg_email_events') }}

{% if is_incremental() %}
WHERE DATE(event_ts) >= DATEADD(day, -3, CURRENT_DATE())
{% endif %}
```

- 按 `date_day` **partition 整块替换**，避免 MERGE 的行级 I/O 开销
- 适合按日期分区的聚合表

#### Z-ORDER 优化

```sql
{{ config(
    post_hook="OPTIMIZE {{ this }} ZORDER BY (account_id, event_ts)"
) }}
```

- `post_hook`：model 跑完之后自动执行
- Z-ORDER 让 account_id + event_ts 的查询跳过更多文件（data skipping）
- 在例如 100M+ daily events 场景下，查询性能提升显著



### 2.10 Jinja 进阶语法（macro 里会用到）

```jinja
{# 注释 #}

{# 变量 #}
{% set my_var = 'hello' %}

{# 条件 #}
{% if condition %}
  ...
{% elif other_condition %}
  ...
{% else %}
  ...
{% endif %}

{# 循环（生成动态 SQL 时非常有用）#}
{% for col in ['email', 'name', 'title'] %}
  UPPER({{ col }}) AS {{ col }}
  {% if not loop.last %},{% endif %}
{% endfor %}

{# 调用其他 macro #}
{{ other_macro(param1, param2) }}

{# run_query — 在编译时执行 SQL，用于动态生成逻辑 #}
{% set results = run_query("SELECT DISTINCT event_type FROM raw.events") %}
{% for row in results.rows %}
  ...
{% endfor %}
```


### 2.11 dbt 常用 packages

在 `packages.yml` 里声明，`dbt deps` 安装：

```yaml
packages:
  - package: dbt-labs/dbt_utils
    version: [">=1.0.0"]
  - package: calogica/dbt_expectations
    version: [">=0.10.0"]    # 建议使用最新兼容版本
```

#### `dbt_utils` 常用函数

| 函数 | 用途 |
|---|---|
| `dbt_utils.generate_surrogate_key(['col1', 'col2'])` | 平台适配的 hash 拼接生成代理键 |
| `dbt_utils.date_spine(...)` | 生成连续日期序列（填补没有数据的日期）|
| `dbt_utils.pivot(...)` | 行转列 |
| `dbt_utils.unpivot(...)` | 列转行 |
| `dbt_utils.get_column_values(ref('model'), 'col')` | 编译时获取列的所有唯一值 |

> 注：`generate_surrogate_key` 底层 hash 函数因平台而异（Snowflake 用 MD5，BigQuery 用 `TO_HEX(MD5(...))`）。面试中说 "hash-based" 比说 "MD5" 更准确。

#### `dbt_expectations`（Great Expectations 风格的 dbt tests）

```yaml
- name: fct_email_events
  data_tests:
    - dbt_expectations.expect_table_row_count_to_be_between:
        min_value: 1000000
        max_value: 200000000
    - dbt_expectations.expect_column_proportion_of_unique_values_to_be_between:
        column_name: event_id
        min_value: 0.99   # 允许极少量重复
```



### 2.12 Exposures & Groups（Governance 话题）

#### Exposures — 声明下游消费者

In dbt, exposures define downstream dependencies of your data models, such as dashboards, reports, or ML applications.

通常放在 `models/` 目录下   

```yaml
# models/marts/_exposures.yml
exposures:
  - name: weekly_revenue_dashboard
    type: dashboard
    maturity: high    # high / medium / low
    owner:
      name: Data Analytics Team
      email: analytics@gmail.com
    depends_on:
      - ref('fct_revenue')
      - ref('dim_accounts')
```

作用：
- 让 dbt 知道哪些外部工具（BI dashboard、ML model、API）消费了你的 model，在 DAG 中可视化端到端依赖。
- 写完之后 dbt docs generate 生成的 DAG 里就能看到：fct_revenue → weekly_revenue_dashboard，从数据一直追踪到最终的 BI 看板。
- 面试场景里的价值是：如果你要改 fct_revenue 的某个字段，DAG 里直接能看到下游有哪些 dashboard 会受影响。

支持：
- dashboard（最常见）
- notebook
- analysis
- ml
- application


#### Groups — 按团队划分所有权
In dbt, groups are used to organize models and assign ownership to teams or domains.  

1. 先定义 group（可以放在 `models/` 下任意 `.yml` 文件中）

  ```yaml
  # models/_groups.yml
  groups:
    - name: data_platform
      owner:
        name: Data Platform Team

  models:
    - name: fct_email_events
      config:
        group: data_platform
  ```
2. 然后在 model 的 YAML 或 config 中指定归属哪个 group：

  ```yaml
  # models/marts/_models.yml
  models:
    - name: fct_email_events
      config:
        group: data_platform

    - name: fct_revenue
      config:
        group: analytics
  ```

Group 的实际作用是配合 access control（v1.5+）。
- 你可以给 model 设 `access: private`，意味着只有同一个 group 内的 model 才能 ref() 它，其他 group 引用会直接报错：

```yaml
models:
  - name: int_account_metrics    # 中间表，不想让别的团队直接依赖
    config:
      group: data_platform
      access: private            # 只有 data_platform group 内部能 ref()
```

面试中围绕 data governance 和 ownership 的话题时可以提及。

---


## Level 3

了解即可（Staff/Principal 层）

### 3.1 dbt Semantic Layer / MetricFlow

- dbt 1.6+ 引入的功能
- 允许在 dbt 里定义 metrics（如 `monthly_active_accounts`），供 BI 工具直接查询
- 好处：metric 定义单一来源，避免 BI 工具里各自定义逻辑不一致
- 2025 年更新：新的 Semantic Layer YAML specification 已发布（Latest release track）
- **面试遇到可以说**："我了解这个方向，我们团队目前还是通过 Gold layer + Domo 消费，但 Semantic Layer 是我们在评估的演进方向"

### 3.2 dbt Mesh（Cross-project ref）

- 大型组织把 dbt 拆成多个 project，各团队独立维护
- `{{ ref('project_name', 'model_name') }}` 可以跨 project 引用
- 适用场景：公司级 core data（accounts, users）独立成一个 project，各业务线 project 依赖它
- 2025 年更新：dbt Catalog 已支持跨项目 lineage 浏览
- **面试遇到可以说**："在 xxx 我们是单 project，但如果平台扩展到多团队，dbt Mesh 是合理的演进路径"

### 3.3 dbt CI/CD

- 每个 PR 自动运行 `dbt build`（Slim CI，只运行变动的 models）
- `dbt build --select state:modified+`：只构建和测试有变化的 models 及其下游
- v1.9 改进：`state:modified` selector 使用 unrendered 配置比较，减少 Slim CI 误判
- **面试提及方式**："我们在 Databricks Workflows 里做 CI，但原理和 dbt Cloud CI 一致——PR 触发、只跑 modified models"

### 3.4 ⭐ dbt Fusion Engine（2025 年最大变化）

2025 年 5 月 dbt Labs 发布了 **dbt Fusion Engine**，这是 dbt 历史上最大的架构变革：

- **完全用 Rust 重写**，不再依赖 Python，与 dbt Core 没有共享代码（除 adapter macros）
- 解析速度比 dbt Core 快 **30 倍**，编译速度快 2 倍
- **原生 SQL 理解**：Fusion 真正解析 SQL 语法和语义，知道列、函数、类型如何在 lineage 中传播
- **State-aware orchestration**：只在数据源有新数据时才运行 job，只构建有变化的 model（平均节省 10% 计算成本）
- **VS Code 扩展**：实时错误检测、live CTE 预览、column-level lineage，无需连接仓库
- 已支持 Snowflake、Databricks、BigQuery、Redshift；Apache Spark 在 beta
- 在 dbt platform（原 dbt Cloud）中，新项目默认使用 Fusion

**dbt Core vs Fusion 定位**：
- dbt Core（Python）仍然是 OSS 版本，继续维护
- dbt Fusion 是新引擎，部分源码公开（ELv2 License）
- Fusion 使用 Arrow Database Connector (ADBC) 驱动，替代了 Python DB-API 适配器

Fusion engine
- it's a ground-up Rust rewrite with native SQL comprehension. 
- The key benefits are much faster parsing, real-time error detection in the IDE, and state-aware orchestration that only rebuilds models when source data changes. 
- At xxx, we're still on dbt Core with Databricks, but Fusion is the direction dbt is heading, especially for teams that want column-level lineage and smarter CI/CD.

### 3.5 dbt MCP Server（AI 集成）

- 2025 年 Coalesce 大会宣布，已 GA
- 允许 AI 助手直接查询 dbt 项目的 metadata — model lineage、macro 定义、test 详情等
- 支持通过 Admin API 自动化 dbt 工作流（触发 runs、管理 artifacts）
- 面试中如果聊到 "future of analytics engineering" 可以提及

### 3.6 dbt Core v1.11 — First-class UDFs

- dbt Core 1.11（2025 年 12 月 GA）引入在 dbt 项目中直接定义和管理 UDF 的能力
- 支持 Python UDFs、默认参数、丰富的配置选项
- 让 dbt 从"纯 SQL transformation" 扩展到管理自定义函数的生命周期
- 面试中不需要深入，但可以作为"我关注最新动态"的信号

---


## 面试高频问题 & 标准答法

### Q1: What's the difference between incremental and table materialization?

**答题框架**：What → Trade-offs → When to use which

> "Table materialization does a full rebuild every run — simple and always correct, but expensive for large datasets. Incremental only processes new or changed data using `is_incremental()` to filter, which is critical at scale. The trade-off is complexity: you have to handle late-arriving data with a lookback window, ensure your `unique_key` is correct, and choose the right incremental strategy — append for immutable events, merge for mutable dimension records, insert_overwrite for partitioned aggregation tables on Databricks. Since dbt 1.9, there's also microbatch which eliminates the need for manual `is_incremental()` logic entirely by splitting processing into independent time-based batches. At Outreach with 100M+ daily events, full table rebuilds weren't viable, so incremental was the default for our fact tables."

---



### Q2: How do you handle schema changes in dbt?

> "dbt provides an `on_schema_change` config with four options: `ignore` which is the default and just silently handles changes, `fail` which errors out, `append_new_columns` which adds new columns automatically, and `sync_all_columns` which fully synchronizes the schema. For additive changes like new columns, `append_new_columns` handles them gracefully. For breaking changes like column renames or type changes, we version the model or do a full refresh with `dbt run --full-refresh`. We also use dbt contracts on our Gold layer to enforce explicit schema agreements — if someone changes a mart model in a way that breaks the contract, the build fails immediately. At Outreach, Debezium CDC introduces schema changes when the upstream Postgres schema evolves, so we have a standard process: schema registry catches the drift first, then we update the dbt model with a PR, then force a full refresh for that model."

---



### Q3: How do you organize a dbt project?

> "We follow a three-layer structure: staging, intermediate, and marts. Staging is 1-to-1 with source tables — only renaming, casting, and basic cleaning, no business logic, no joins, materialized as views. Intermediate handles cross-source joins and complex business logic, scoped to analysts who need it. Marts are the consumption layer — `dim_` tables for dimensions, `fct_` for facts, incremental for high-volume tables. This separation means staging can change without breaking marts, and mart logic stays readable without getting tangled in source-specific quirks."

---



### Q4: Walk me through how you built the SCD Type 2 macro.

> "At Outreach, dimensions like accounts and contacts evolve — ownership changes, tier upgrades, industry reclassification. We needed to track history rather than overwrite. I built a parameterized dbt macro that takes the source relation, the business key, a timestamp column, and the list of tracked columns. The macro generates a row hash by concatenating and hashing the tracked columns, then compares incoming records against the current snapshot using that hash. If the hash differs, it marks the existing row as `is_current = False` with an `effective_end_date`, and inserts the new version as `is_current = True`. Surrogate keys use `dbt_utils.generate_surrogate_key()` combining the business key and effective start date. The macro got adopted across all our dimension models — it cut the per-model implementation time significantly and eliminated inconsistencies. I should note dbt also has built-in snapshots for SCD Type 2 which got a major upgrade in v1.9 — for standard use cases those are simpler, but our macro offered more control over hash-based detection and custom surrogate keys."

---



### Q5: How do you test data quality in dbt?

> "We use a layered approach with three types of testing. First, unit tests validate SQL logic before models build — I define mock inputs and expected outputs in YAML, and dbt checks my transformation logic against those, great for catching edge cases in complex calculations. Second, at the model level, dbt's built-in generic data tests cover uniqueness, not-null, accepted values, and referential integrity — these run on every PR via `dbt build`. For more sophisticated checks, we use `dbt_expectations` for things like row count thresholds and column distribution checks. We set `store_failures: true` so failed test records are written to a dedicated schema, making it easy to diagnose which exact rows caused a failure. Third, at the pipeline level, Great Expectations acts as a data contract at the ingestion boundary — catching schema drift and statistical anomalies before bad data reaches dbt at all."

---



### Q6: How does dbt fit into your overall pipeline?

> "dbt sits entirely within the transformation layer. Upstream, Debezium CDC captures changes from Postgres and publishes to Kafka topics; Spark Structured Streaming consumes those into our Delta Lake Bronze layer. From there, dbt handles all transformations — staging models clean and standardize the raw Delta tables, intermediate models join and apply business logic, and mart models produce the Gold layer for BI dashboards in Domo and feature tables for our ML models via MLflow. Orchestration is through Databricks Workflows, which runs dbt as a task in the pipeline DAG. We use `dbt build` rather than separate run and test steps, so tests are interleaved in the DAG and a failing test blocks downstream models from building."

---



### Q7: What's the microbatch strategy and when would you use it?

> "Microbatch is a new incremental strategy introduced in dbt Core 1.9, designed for large time-series datasets. Instead of writing one big SQL query with `is_incremental()` logic, you configure `event_time`, `batch_size`, and optionally `lookback`, and dbt automatically splits processing into independent time-based batches — one SQL query per batch. Each batch is idempotent and can be retried independently, which is a huge win for reliability. You can also do targeted backfills with `--event-time-start` and `--event-time-end` instead of a full refresh. The trade-off is you need a reliable time column, and all upstream models need `event_time` configured to avoid full table scans. I'd use it for our 100M+ daily event tables at Outreach where traditional incremental with lookback windows was getting complex to maintain."

---



### Q8: What's the difference between data tests and unit tests in dbt?

> "Data tests validate data quality after a model builds — things like uniqueness, not-null, accepted values, referential integrity. They run against real data in the warehouse. Unit tests, introduced in dbt 1.8, validate your SQL transformation logic before the model builds. You define static mock inputs and expected outputs in YAML, and dbt checks if your SQL produces the right result. Think of data tests as integration tests on your data, and unit tests as logic tests on your code. In practice, I'd use unit tests for models with complex calculations or conditional logic where edge cases matter."

---



### Q9: How do you handle hard deletes in dimension tracking?

> "dbt snapshots now support a `hard_deletes` config with three modes: `ignore` which does nothing, `invalidate` which marks deleted records by setting their `dbt_valid_to`, and `new_record` which inserts a new row with a `dbt_is_deleted` flag. For our account dimensions at Outreach, `invalidate` was sufficient — when an account is deleted from the source, the snapshot closes out that record's validity window. For audit-heavy environments where you need an explicit deletion record in the history, `new_record` is the better choice."

---


## 快速记忆卡片

```text
dbt 的本质：ELT 里的 T，仓库内部的 SQL 管理工具

五种 materialization：
  view → table → incremental → ephemeral → snapshot
  （越往右越复杂）

六种 incremental strategy（v1.9+）：
  append → merge → delete+insert → insert_overwrite → microbatch

传统 Incremental 三要素：
  1. unique_key（去重 key）
  2. is_incremental()（filter 条件）
  3. {{ this }}（引用自身，查 watermark）

Microbatch 三要素（v1.9+）：
  1. event_time（时间列）
  2. batch_size（batch 粒度：day/hour）
  3. lookback（late-arriving data 回看窗口）
  ⚠️ 不需要 is_incremental() 和 {{ this }}！

`model/`分层结构：
  raw → staging（clean）→ intermediate（join）→ marts（consume）

三种测试（v1.8+）：
  unit_tests → SQL 逻辑验证（mock 输入 → 期望输出）
  data_tests → 数据质量（unique, not_null, accepted_values）
  Great Expectations → ingestion boundary 数据契约

Snapshot v1.9 升级：
  YAML 配置 → hard_deletes（ignore/invalidate/new_record）
  → dbt_valid_to_current → snapshot_meta_column_names

Macro 两大使用场景：
  1. SCD Type 2（历史追踪，hash-based change detection）
  2. Deduplication（去重）

dbt build = DAG 交错执行 seed → snapshot → run → test
  test 失败阻断下游，比 dbt run + dbt test 更安全

dbt Contracts = 强制 schema 约定（v1.5+）
  contract.enforced: true → schema 不符直接报错

dbt Fusion Engine（2025）：
  Rust 重写 → 30x 解析速度 → SQL 理解 → State-aware orchestration

on_schema_change 四个值：
  ignore → fail → append_new_columns → sync_all_columns
```



# Cheat Sheet

<img src='./pic/dbt_cheat_sheet.jpeg' width=800>